![Notebook 3](images/architecture-detailed.svg)

# 3. Let AgentCore find the bug, then verify its fix

> Deploy an agent with a bug, let AgentCore run a dataset against it, cluster the
> failures and rewrite the prompt. **Then A/B the rewrite on live traffic, because a
> generated fix is a hypothesis.**

⚠️ This notebook creates billable resources and takes about 90 minutes. The last cell
tears everything down. Notebooks 1 and 2 need neither.

📖 Story and gotchas: [`README.md`](README.md)
💻 Code this notebook imports: [`helpers/`](helpers/) (`agentcore_project.py`, `agentcore_loop.py`, `research_checks.py`)
⬅️ Previous: [`02_score_traces_with_agentcore_evaluations.ipynb`](02_score_traces_with_agentcore_evaluations.ipynb)

---

## Prerequisite

The [AgentCore CLI](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-get-started-cli.html)
1.2 or later deploys everything below: `npm install -g @aws/agentcore` (Node.js 20 or
later). It also turns on CloudWatch Transaction Search, without which there are no spans.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, "helpers")   # the modules this notebook imports
os.environ.setdefault("AWS_REGION", "us-west-2")
REGION = os.environ["AWS_REGION"]
WORK = Path("agentcore_eval_artifacts").resolve()
print(subprocess.run(["agentcore", "--version"], capture_output=True, text=True).stdout.strip())

## Deploy an agent with a bug

"Production" runs `LOOPING_COORDINATOR_PROMPT`: the coordinator with its two filesystem
rules removed. It hunts for the analyst's chart in a filesystem it cannot see and loops
until LangGraph's recursion ceiling. One CLI project holds the runtime (`helpers/` as
it is), a gateway, and two online evaluations: `GoalSuccessRate` for A/B tests, and
`FailureAnalysis` plus the multi-dimension evaluator from notebook 2 for monitoring.

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient
from agentcore_project import deploy
from research_checks import RESEARCH_QUALITY

quality = EvaluationClient(region_name=REGION).create_evaluator_and_wait(
    evaluator_name=f"ResearchQuality{int(time.time())}", level="SESSION", evaluator_config=RESEARCH_QUALITY)
project = deploy(name="ResearchEval", code_dir=Path("helpers"), entrypoint="runtime_entrypoint.py",
                 region=REGION, work_dir=WORK, env={"COORDINATOR_PROMPT": "looping"},
                 monitor_evaluators=["Builtin.FailureAnalysis", quality["evaluatorArn"]])
print(project.runtime_arn)

## Find out what failed

A dataset holds the scenarios in `research_checks.py`, some designed to make the agent
fail. One batch evaluation runs every scenario against the runtime, then scores each
session with `GoalSuccessRate` and clusters the failures with `FailureAnalysis`.
AgentCore invokes the agent itself, so there is no traffic loop here.

In [ ]:
from agentcore_loop import diagnose, publish_dataset, root_causes
from research_checks import FAILURE_SCENARIOS, SUCCESS_SCENARIOS

dataset_id = publish_dataset(region=REGION, name=f"research_{int(time.time())}",
                             scenarios=SUCCESS_SCENARIOS + FAILURE_SCENARIOS)
detail = diagnose(region=REGION, endpoint_arn=project.endpoint_arn, service_name=project.service_name,
                  log_group=project.log_group, dataset_id=dataset_id)
print(detail["status"], {k: v for k, v in detail["evaluationResults"].items() if k.startswith("numberOf")})
for s in detail["evaluationResults"]["evaluatorSummaries"]:
    print(f"  {s['evaluatorId']:<26} {s['statistics'].get('averageScore')}")
for rc in root_causes(detail):
    print(f"\n  [{rc['sessions']} sessions] {rc['insight']}\n    {rc['root_cause']}")

## Get a prompt rewrite

The production prompt goes in a configuration bundle. The recommendation reads it from
there and writes its rewrite as a **new version of the same bundle**: that version is the
A/B treatment, with no copying. It optimizes `GoalSuccessRate`, not the root causes
above, so compare the two.

In [ ]:
from agentcore_loop import make_bundle, recommend_prompt
from research_agent import COMPANIES, COORDINATOR_PROMPT, LOOPING_COORDINATOR_PROMPT

names = ", ".join(c.name for c in COMPANIES)
S = int(time.time())
control = make_bundle(region=REGION, runtime_arn=project.runtime_arn, name=f"Loop{S}",
                      prompt=LOOPING_COORDINATOR_PROMPT.format(company_list=names), note="production")
generated, why = recommend_prompt(region=REGION, control=control, runtime_arn=project.runtime_arn,
                                  batch_evaluation_arn=detail["batchEvaluationArn"])
hits = [s for s in ("read_file", "isolated", "working directory", "/tmp") if s in generated.prompt.lower()]
print(f"{len(control.prompt)} -> {len(generated.prompt)} chars. Filesystem guidance: {hits or 'none'}\n")
print(why[:1200])

## A/B the fixes on live traffic

The gateway assigns each new session to an arm and hands the runtime that arm's bundle
version, so nothing is redeployed. Two rounds against the same control: the generated
rewrite, then the known human fix, so you see what a fix that works looks like next to
one that may not. A gateway runs one test at a time, so each round stops its test.

In [ ]:
from agentcore_project import ab_test
from research_checks import CHART_PROMPT, metrics_found

human = make_bundle(region=REGION, runtime_arn=project.runtime_arn, name=f"Fix{S}",
                    prompt=COORDINATOR_PROMPT.format(company_list=names), note="human fix")
rounds = {label: ab_test(project, name=f"{label}{S}", control=control, treatment=t, prompt=CHART_PROMPT, n=12)
          for label, t in (("generated", generated), ("human", human))}

print(f"  {'round':<10}{'arm':<10}{'GoalSuccess':>12}{'completed':>11}{'recursion':>11}{'metrics_of_9':>14}")
for label, got in rounds.items():
    for arm, row in got["arms"].items():
        m9 = sum(map(metrics_found, row["answers"])) / len(row["answers"]) if row["answers"] else 0
        print(f"  {label:<10}{('control' if arm == 'C' else 'treatment'):<10}{row['goal_success']:>12.2f}"
              f"{row['completed']:>6}/{row['sessions']:<4}{row['recursion']:>11}{m9:>14.1f}")

## What monitoring saw

The `monitor` evaluation scored the same live traffic. Each dimension of the quality
evaluator is its own result, and its own CloudWatch metric (`<evaluator>/<dimension>`)
you can alarm on. `FailureAnalysis` there also feeds Agent Insights, which aggregate root
causes per agent over time (`ListAgentInsights`). Expect a `ValidationException` on the
looping sessions: FailureAnalysis rejects a session above 250,000 input tokens.

In [ ]:
from agentcore_project import monitoring

for r in monitoring(project):
    print(f"  {r['evaluator']:<48} {r.get('result', ''):<20} {r['sessions']}")

## What to measure with what

| Question | Instrument |
|---|---|
| Is the answer right? | `Correctness`, `Helpfulness`, ground-truth checks |
| Did it achieve the goal? | `GoalSuccessRate` |
| Why did it fail? | `FailureAnalysis` root causes and Agent Insights |
| Were the figures read, not recalled? | the multi-dimension evaluator |
| Right tools, right args? | `ToolSelectionAccuracy`, `ToolParameterAccuracy` |
| Expected path? | `TrajectoryInOrderMatch`, leaf tools only |
| Wasting work? | trajectory shape from spans |
| Completing at all? | completion rate and recursion failures |

Two things this loop taught us the hard way. The built-in evaluators scored **1.0 in both
arms** of an A/B where one arm used 16% fewer tool calls, so they cannot see wasted work.
And targeting `GoalSuccessRate` made one run's rewrite *add* a confirmation gate and
another's *remove* one, because asking scores as less complete than answering.

To ship the winner, add a gateway rule that applies its bundle version to all traffic:
see [Route traffic to the winner](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-config-bundle.html).

## Tear down

Log groups are kept on purpose: they hold the spans every score above references.

In [ ]:
CLEAN = False  # set True and re-run

if not CLEAN:
    print(f"CLEAN is False. Would remove everything in {project.root}, bundles")
    print(f"{control.bundle_id} and {human.bundle_id}, and evaluator {quality['evaluatorId']}.")
else:
    from agentcore_loop import delete_bundles
    from agentcore_project import teardown
    teardown(project)
    delete_bundles(region=REGION, bundle_ids=[control.bundle_id, human.bundle_id])
    EvaluationClient(region_name=REGION).delete_evaluator_and_wait(evaluator_id=quality["evaluatorId"])